This notebook downloads vacancies from rabota.by published for the last month and saves them to .json file with list of dictionaries.

In [6]:
import sys
import time
import json
import requests
import datetime

from functools import wraps
from requests.exceptions import HTTPError, ConnectionError, Timeout
from random_user_agent.user_agent import UserAgent
from random_user_agent.params import SoftwareName, OperatingSystem
from tqdm.auto import trange


SOFTWARE_NAMES = [SoftwareName.CHROME.value]
OPERATING_SYSTEMS = [OperatingSystem.WINDOWS.value, OperatingSystem.LINUX.value]
USER_AGENT = UserAgent(software_names=SOFTWARE_NAMES, operating_systems=OPERATING_SYSTEMS, limit=100)

VACANCY_URL = "https://api.hh.ru/vacancies/{}"
VACANCY_PAGE_URL = "https://api.hh.ru/vacancies?area={}&industry={}&period={}&page={}&per_page=100"
VACANCY_PAGE_URL_WITHOUT_INDUSTRY = "https://api.hh.ru/vacancies?area={}&period={}&page={}&per_page=100"
VACANCY_PAGE_URL_WITH_DATES = "https://api.hh.ru/vacancies?area={}&industry={}&date_from={}&date_to={}&page={}&per_page=100"
VACANCY_PAGE_URL_WITH_DATES_WITHOUT_INDUSTRY = "https://api.hh.ru/vacancies?area={}&date_from={}&date_to={}&page={}&per_page=100"


def download(get_url):
    @wraps(get_url)
    def wrapper(*args, timeout=10, requests_interval=10, max_requests_number=100, break_reasons=None):
        """
        :param int requests_interval: time interval between requests (sec.)
        :param int max_requests_number: maximum number of requests
        :param list break_reasons: list of reasons
        """
        url = get_url(*args)
        break_reasons = set() if break_reasons is None else set(break_reasons)

        for _ in range(max_requests_number):
            try:
                request = requests.get(url, headers={'User-Agent': USER_AGENT.get_random_user_agent()}, timeout=timeout)
                request.raise_for_status()
            except ConnectionError as connection_error:
                print(f"Connection error occurred: {connection_error}", file=sys.stderr)
            except Timeout as time_out:
                print(f"Timeout error occurred: {time_out}", file=sys.stderr)
            except HTTPError as http_error:
                print(f"HTTP error occurred: {http_error}", file=sys.stderr)
                if request.reason in break_reasons:
                    break
            else:
                return request.content

            print(f"A second request to the {url} will be sent in {requests_interval} seconds")
            time.sleep(requests_interval)

        raise HTTPError(f"Page on this {url} has not been downloaded")
    return wrapper


def load_json(get_content):
    @wraps(get_content)
    def wrapper(*args, **kwargs):
        return json.loads(get_content(*args, **kwargs))
    return wrapper


@load_json
@download
def vacancy_search_page(area_id, industry_id, search_period, num_page):
    """
    :param area_id: area identifier from https://api.hh.ru/areas
    :param industry_id: industry identifier from https://api.hh.ru/industries
    :param search_period: the number of days for search or dict with search period
    :param num_page: page number
    :return: str
    """
    if isinstance(search_period, int):
        if industry_id is None:
            return VACANCY_PAGE_URL_WITHOUT_INDUSTRY.format(area_id, search_period, num_page)
        else:
            return VACANCY_PAGE_URL.format(area_id, industry_id, search_period, num_page)
    elif isinstance(search_period, dict):
        if industry_id is None:
            return VACANCY_PAGE_URL_WITH_DATES_WITHOUT_INDUSTRY.format(area_id, search_period['date_from'], search_period['date_to'], num_page)
        else:
            return VACANCY_PAGE_URL_WITH_DATES.format(area_id, industry_id, search_period['date_from'], search_period['date_to'], num_page)
    else:
        raise RuntimeError(f'Invalid type of the search_period. Expected int or dict, got {type(search_period)}')


def vacancy_ids_from_small_period(area_id, industry_id, search_period, num_pages, **kwargs):
    """
    :param area_id: area identifier from https://api.hh.ru/areas
    :param industry_id: industry identifier from https://api.hh.ru/industries
    :param search_period: the number of days for search
    :param num_pages: number pages for download
    :return: list
    """
    if num_pages is None:
        num_pages = 19

    ids = []
    for num_page in range(num_pages):
        page = vacancy_search_page(area_id, industry_id, search_period, num_page, **kwargs)

        if not page["items"]:
            break

        ids.extend([item["id"] for item in page["items"]])

    return list(set(ids))


def vacancy_ids(area_id, industry_id=None, search_period=30, num_pages=None, step=30, **kwargs):
    """
    :param area_id: area identifier from https://api.hh.ru/areas
    :param industry_id: industry identifier from https://api.hh.ru/industries
    :param search_period: the number of days for search
    :param num_pages: number pages for download
    :param step: the number of days for search in one step
    :return: list
    """
    date_to = datetime.datetime.now()
    ids = []
    for i in trange(0, search_period, step):
        days = min(search_period - i, step) - 1
        date_from = date_to - datetime.timedelta(days=days)

        search_period_with_dates = {'date_from': date_from.strftime('%Y-%m-%d'), 'date_to': date_to.strftime('%Y-%m-%d')}
        ids.extend(vacancy_ids_from_small_period(area_id, industry_id, search_period_with_dates, num_pages, **kwargs))
        ids = list(set(ids))

        date_to = date_from - datetime.timedelta(days=1)

    return ids

In [47]:
vacancy_ids = vacancy_ids(16, step=1)
len(vacancy_ids)

  0%|          | 0/30 [00:00<?, ?it/s]

29708

In [73]:
import aiohttp
import asyncio
import time

from tqdm import tqdm
from tqdm.asyncio import tqdm_asyncio


async def get_vacancy(session, id_):
    try:
        async with session.get(VACANCY_URL.format(id_), 
                               headers={'User-Agent': USER_AGENT.get_random_user_agent()}, 
                               timeout=20) as resp:
            return await resp.json()
    except Exception as e:
        pass

async def async_gather_http_get(vacancy_ids, received_ids):
    async with aiohttp.ClientSession() as session:
        tasks = []
        for id_ in tqdm(vacancy_ids):
            if id_ in received_ids:
                continue
            tasks.append(asyncio.create_task(get_vacancy(session, id_)))

        return await tqdm_asyncio.gather(*tasks)


received_ids = set()
vacancies = []

lower_bound_for_time_to_wait = 5
upper_bound_for_time_to_wait = 20
required_increase = 100
cur_time_to_wait = None

while True:
    prev_len_of_received = len(received_ids)

    cur_vacancies = await async_gather_http_get(vacancy_ids, received_ids)
    for vacancy in cur_vacancies:
        if isinstance(vacancy, dict) and 'id' in vacancy:
            received_ids.add(vacancy['id'])
            vacancies.append(vacancy)

    print(f'Number of received: {len(received_ids)}/{len(vacancy_ids)}')

    if cur_time_to_wait is not None:
        if len(received_ids) - prev_len_of_received < required_increase:
            print(f'Setting lower_bound_for_time_to_wait to {cur_time_to_wait}...')
            lower_bound_for_time_to_wait = cur_time_to_wait
        else:
            print(f'Setting upper_bound_for_time_to_wait to {cur_time_to_wait}...')
            upper_bound_for_time_to_wait = cur_time_to_wait
    
    if len(received_ids) != len(vacancy_ids):
        cur_time_to_wait = (lower_bound_for_time_to_wait + upper_bound_for_time_to_wait) / 2
        print(f'Timeout for {cur_time_to_wait} seconds...')
        time.sleep(cur_time_to_wait)
    else:
        break


100%|██████████| 29708/29708 [00:00<00:00, 1095432.78it/s]

100%|██████████| 16/16 [00:10<00:00,  1.53it/s]


Number of received: 29692/29708
Setting lower_bound_for_time_to_wait to 11.000121951336151...
Timeout for 11.000121951336151 seconds...



100%|██████████| 29708/29708 [00:00<00:00, 1275651.71it/s]

100%|██████████| 16/16 [00:10<00:00,  1.53it/s]


Number of received: 29692/29708
Setting lower_bound_for_time_to_wait to 11.000121951336151...
Timeout for 11.000121951336151 seconds...



100%|██████████| 29708/29708 [00:00<00:00, 1022269.12it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

CancelledError: 

In [78]:
import json

with open('../data/vacancies.json', 'w') as fout:
    json.dump(vacancies, fout)